# Custom surface site analysis

Edit **Cell 2 only**, then **Run All**.


In [1]:
# Imports (DO NOT EDIT)
import sys, os
sys.path.append(".")
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../..')))

from ase import Atom, Atoms
import copy
from ase.io import read, write, Trajectory
from acat.settings import CustomSurface
from acat.adsorption_sites import SlabAdsorptionSites
from ase.build import surface,fcc111
from ase.visualize import view
from site_analysis import *

In [2]:

# Load pre-made slab
slab=read('/home/shikim/pynta/pynta/preprocessing/custom_surfaces/pd332.xyz')
nslab = len(slab)
adsorbate_height = 2    
site_bond_cutoff = 1.5
view(slab,viewer='x3d')


In [3]:
cas = SlabAdsorptionSites(slab, "fcc332", composition_effect=True)  # ACAT has surface, custom does not find them all!
single_geoms, single_sites_lists = generate_unique_sites(
        slab,
        cas.get_sites(),
        nslab,
        site_bond_cutoff,
        adsorbate_height
    )

In [4]:

# Extract site graphs
admols, geom_indices = classify_all_sites(
    single_geoms, single_sites_lists
)


In [5]:

# Graph isomorphism clustering
iso_mat, clusters = cluster_isomorphic_graphs(admols)


In [6]:

# Pairwise strict isomorphism (PRINT)
print("Pairwise strict isomorphism:")
for i in range(len(admols)):
    for j in range(i + 1, len(admols)):
        print(f"{i} vs {j} =", iso_mat[i, j])


Pairwise strict isomorphism:
0 vs 1 = False
0 vs 2 = False
0 vs 3 = False
0 vs 4 = False
0 vs 5 = False
0 vs 6 = False
0 vs 7 = False
0 vs 8 = False
0 vs 9 = False
0 vs 10 = False
0 vs 11 = False
0 vs 12 = False
0 vs 13 = False
0 vs 14 = False
0 vs 15 = False
0 vs 16 = False
0 vs 17 = False
0 vs 18 = False
0 vs 19 = False
0 vs 20 = False
0 vs 21 = False
0 vs 22 = False
0 vs 23 = False
0 vs 24 = False
0 vs 25 = False
0 vs 26 = False
0 vs 27 = False
0 vs 28 = False
0 vs 29 = False
0 vs 30 = False
0 vs 31 = False
0 vs 32 = False
0 vs 33 = False
0 vs 34 = False
0 vs 35 = False
0 vs 36 = False
0 vs 37 = False
0 vs 38 = False
0 vs 39 = False
0 vs 40 = False
1 vs 2 = True
1 vs 3 = True
1 vs 4 = True
1 vs 5 = True
1 vs 6 = True
1 vs 7 = True
1 vs 8 = True
1 vs 9 = True
1 vs 10 = False
1 vs 11 = False
1 vs 12 = True
1 vs 13 = True
1 vs 14 = True
1 vs 15 = False
1 vs 16 = False
1 vs 17 = False
1 vs 18 = True
1 vs 19 = True
1 vs 20 = True
1 vs 21 = True
1 vs 22 = True
1 vs 23 = True
1 vs 24 = Tru

In [7]:

# Assuming single_sites_lists and clusters are defined# Assuming single_sites_lists and clusters are defined
updated_sites, site_mapping = update_site_labels(single_sites_lists, clusters)

Clusters structure: {0: [0], 1: [1, 2, 3, 4, 5, 6, 7, 8, 9, 12, 13, 14, 18, 19, 20, 21, 22, 23, 24, 28, 31, 33, 34, 35, 36, 37, 38, 39, 40], 10: [10], 11: [11, 17, 32], 15: [15, 16, 26, 27], 25: [25], 29: [29], 30: [30]}
Single sites lists structure: [[{'site': 'ontop', 'surface': 'fcc332', 'morphology': 'terrace', 'position': array([-2.88511436e-17,  1.84855763e+00,  2.72094995e+01]), 'normal': array([ 0.13380384, -0.11133153,  0.98473439]), 'indices': (8,), 'composition': 'Pd', 'subsurf_index': None, 'subsurf_element': None, 'label': None, 'topology': array([8])}], [{'site': 'bridge', 'surface': 'fcc332', 'morphology': 'sc-tc-h', 'position': array([-1.36902692,  2.0917889 , 27.42154358]), 'normal': array([ 0.13290678, -0.11058512,  0.98493996]), 'indices': (8, 9), 'composition': 'PdPd', 'subsurf_index': None, 'subsurf_element': None, 'label': None, 'topology': array([8, 9])}], [{'site': 'fcc', 'surface': 'fcc332', 'morphology': 'sc-tc-h', 'position': array([-1.22561794,  2.89367733, 

# detect defects

In [10]:
slab_xyz = read("/home/shikim/pynta-code/site/pynta_preprocessing_test/Pt_defect/Pt111_vacancy.xyz")
n_layers = 4
surface = CustomSurface(slab_xyz, n_layers=n_layers)
nslab = len(slab_xyz)

In [12]:
slab_xyz = "/home/shikim/pynta-code/site/pynta_preprocessing_test/Pt_defect/Pt111_vacancy.xyz"
defect_sites, slab = detect_defect_sites_from_xyz(
    xyz_path=slab_xyz,
    nslab=nslab,
    grid_n=90,             # grid resolution (bigger = more accurate, slower)
    z_tol=None,            # auto top-layer thickness
    hole_thresh_factor=0.55,  # stricter -> fewer "holes"
    cluster_factor=0.60,      # clustering distance in units of nn distance
    max_sites=20
)

print(f"Detected {len(defect_sites)} defect sites:")
for s in defect_sites:
    print(s["name"], s["position"])

Detected 2 defect sites:
defect_3 [ 5.70911367  7.27334107 14.93493181]
defect_7 [10.80485149  4.82165307 14.93493181]


In [20]:
geoms, bond_params, sites_lists = generate_unique_sites_vacancy(
    geo=slab_xyz,
    sites=cas.get_sites(),
    slab= slab_xyz,
    nslab=nslab,
    site_bond_params_list=[],
    sites_list=[]
)


FileNotFoundError: [Errno 2] No such file or directory: 'slab_vacancy.xyz'